# E07

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from copy import deepcopy
from pathlib import Path

# Generate the caustic training-condition dataset used by the SAM2 transfer notebooks.
ALLOW_LOCAL_GENERATION = False
RESET_EXISTING = False
RESUME_EXISTING = True
RANDOM_SEED = 12345
TRAINING_NUM_FRAMES = 60
DEFAULT_FPS = 40.0
CONDITION_SET_NAME = 'E07_caustic_training_conditions'
EXPECTED_CONDITION_COUNT = 72
BUILD_SAM2_DERIVED_DATASET = True
SAM2_TRAIN_NUM_FRAMES = 3
SAM2_VAL_FRACTION = 0.15
SAM2_VAL_RANDOM_SEED = 42
SAM2_LOSS_SEMANTICS_VERSION = 'loss_weight_positive_target_v2'
SAM2_MIN_VALID_FRAMES = SAM2_TRAIN_NUM_FRAMES
SAM2_CONVERSION_RESIZE_LONG_SIDE_PX = None


import shutil
import tempfile
import zipfile


SOURCE_ZIP_NAME = 'syniscopy_source.zip'
IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([
        Path('/content/supplemental'),
        Path('/content/SyniscopySupplemental'),
    ])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists() and (
            (resolved / SOURCE_ZIP_NAME).exists() or (resolved.parent / 'codebase').is_dir()
        ):
            return resolved
    raise RuntimeError(
        'Syniscopy supplemental folder not found. Upload the supplemental folder to Drive as MyDrive/supplemental, '
        'including syniscopy_source.zip.'
    )


def prepare_syniscopy_source(supplemental_root: Path) -> Path:
    source_root = supplemental_root.parent
    if (source_root / 'codebase').is_dir():
        return source_root
    zip_path = supplemental_root / SOURCE_ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Missing {zip_path}. Run python supplemental/package_experiments_for_colab.py before uploading supplemental/.'
        )
    extract_dir = Path('/content/syniscopy_source') if IN_COLAB else Path(tempfile.gettempdir()) / 'syniscopy_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    if (extract_dir / 'codebase').is_dir():
        return extract_dir
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'codebase').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Could not find a single codebase/ root after extracting {zip_path}.')
    return candidates[0]


SUPPLEMENTAL_ROOT = find_supplemental_root()
ROOT = prepare_syniscopy_source(SUPPLEMENTAL_ROOT)
CODEBASE = ROOT / 'codebase'
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E07' / 'dataset' / 'raw_syniscopy'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for module_path in [str(CODEBASE), str(ROOT)]:
    if module_path in sys.path:
        sys.path.remove(module_path)
    sys.path.insert(0, module_path)

import importlib
importlib.invalidate_caches()
SYNISCOPY_MODULES = [
    'camera_noise', 'config', 'counterfactual_packets', 'dataset_generator',
    'dataset_schema', 'imaging_base', 'imaging_model', 'kohler_imaging',
    'main', 'mask_generation', 'materials', 'metadata', 'optics',
    'param_schema', 'param_utils', 'particle_model', 'particle_specs',
    'postprocessing', 'presets', 'pupil_sampling', 'rendering',
    'single_frame_viewer', 'substrate', 'substrate_fitting', 'substrate_pattern',
    'supervision_policy', 'trackability', 'trajectory'
]
for name in SYNISCOPY_MODULES:
    if name in sys.modules:
        del sys.modules[name]

CONDITION_PATH = SUPPLEMENTAL_ROOT / 'E07_training_conditions.json'
CONDITIONS = json.loads(CONDITION_PATH.read_text(encoding='utf-8'))
if len(CONDITIONS) != EXPECTED_CONDITION_COUNT:
    raise RuntimeError(f'Expected {EXPECTED_CONDITION_COUNT} E07 training conditions, got {len(CONDITIONS)} from {CONDITION_PATH}')
families = Counter(str(c.get('approved_family')) for c in CONDITIONS)
print('repo:', ROOT)
print('condition file:', CONDITION_PATH)
print('output dataset:', OUTPUT_DIR)
print('conditions:', len(CONDITIONS), dict(families))
print('first:', {k: CONDITIONS[0].get(k) for k in ['condition_id', 'approved_family', 'name', 'z_nm', 'background_counts', 'image_size_px', 'pixel_size_nm']})
print('last:', {k: CONDITIONS[-1].get(k) for k in ['condition_id', 'approved_family', 'name', 'z_nm', 'background_counts', 'image_size_px', 'pixel_size_nm']})


def present(value) -> bool:
    return value is not None and value != ''


def as_float(value, default: float) -> float:
    return float(value) if present(value) else float(default)


def as_int(value, default: int) -> int:
    return int(value) if present(value) else int(default)


def particle(z_nm: float, side_nm: float) -> dict:
    return {
        'name': 'gold_50nm_target',
        'motion': {
            'hydrodynamic_diameter_nm': 50.0,
            'initial_position_nm': [0.5 * side_nm, 0.5 * side_nm, float(z_nm)],
        },
        'signal_multiplier': 1.0,
        'components': [{
            'shape': 'sphere',
            'offset_nm': [0.0, 0.0, 0.0],
            'diameter_nm': 50.0,
            'material': 'Gold',
            'refractive_index': None,
            'signal_multiplier': 1.0,
            'material_properties': None,
        }],
    }


def build_params(video_index: int, rng) -> dict:
    from config import PARAMS

    condition = CONDITIONS[int(video_index)]
    family = str(condition.get('approved_family') or 'recovered')
    image_size = as_int(condition.get('image_size_px'), 256)
    pixel_size_nm = as_float(condition.get('pixel_size_nm'), 116.0)
    fps = as_float(condition.get('fps'), DEFAULT_FPS)
    z_nm = as_float(condition.get('z_nm'), 0.0)
    side_nm = image_size * pixel_size_nm
    background_counts = as_float(condition.get('background_counts'), 6500.0)
    exposure_ms = as_float(condition.get('exposure_ms'), 1000.0 / fps)
    na = as_float(condition.get('na'), 1.2)
    wavelength_nm = as_float(condition.get('wavelength_nm'), 520.0)
    viscosity_pa_s = as_float(condition.get('viscosity_pa_s'), 0.0015)
    empirical_std = as_float(condition.get('empirical_std'), 0.004)
    empirical_gradient = as_float(condition.get('empirical_gradient'), 0.002)
    drift_velocity = condition.get('drift_nm_s') if present(condition.get('drift_nm_s')) else [0.0, 0.0, 0.0]
    vibration_nm = as_float(condition.get('vibration_nm'), 0.0)

    params = deepcopy(PARAMS)
    params.update({
        'imaging_model': 'interferometric',
        'image_size_pixels': image_size,
        'pixel_size_nm': pixel_size_nm,
        'pupil_samples': image_size,
        'fps': fps,
        'num_frames': TRAINING_NUM_FRAMES,
        'duration_seconds': TRAINING_NUM_FRAMES / fps,
        'exposure_time_ms': exposure_ms,
        'wavelength_nm': wavelength_nm,
        'numerical_aperture': na,
        'refractive_index_medium': 1.33,
        'refractive_index_immersion': 1.518,
        'viscosity_Pa_s': viscosity_pa_s,
        'background_intensity': background_counts,
        'read_noise_counts': 1.0,
        'camera_gain_e_per_count': 1.0,
        'shot_noise_enabled': True,
        'gaussian_noise_enabled': True,
        'fixed_pattern_gain_std': 0.0,
        'fixed_pattern_offset_counts': 0.0,
        'hot_pixel_fraction': 0.0,
        'scan_line_noise_counts': 0.0,
        'sample_environment_enabled': True,
        'sample_environment_pattern_enabled': False,
        'sample_environment_pattern': 'none',
        'empirical_background_enabled': True,
        'empirical_background_model': 'multiscale_gaussian_field',
        'empirical_background_relative_std': empirical_std,
        'empirical_background_gradient_relative_strength': empirical_gradient,
        'background_subtraction_method': str(condition.get('background_method') or 'reference_frame'),
        'motion_blur_enabled': True,
        'motion_blur_subsamples': 4,
        'drift_velocity_nm_per_s': drift_velocity,
        'vibration_jitter_std_nm': vibration_nm,
        'vibration_include_axial': False,
        'z_stack_step_nm': 50.0,
        'initial_z_span_nm': max(4000.0, abs(z_nm) * 2.0 + 1000.0),
        'z_motion_constraint_model': 'unconstrained',
        'particles': [particle(z_nm, side_nm)],
        'rotational_diffusion_enabled': False,
        'mask_generation_enabled': True,
        'mask_outer_ring_count': 1,
        'mask_max_area_fraction': 0.35,
        'supervision_target': 'mask_supported',
        'supervision_support_factors': ['temporal', 'signal', 'information'],
        'supervision_temporal_support_enabled': True,
        'supervision_signal_support_enabled': True,
        'supervision_information_support_enabled': True,
        'save_frame_sequence': True,
        'save_raw_frame_views': True,
    })
    return params


if IN_COLAB or ALLOW_LOCAL_GENERATION:
    from dataset_generator import generate_dataset

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    condition_manifest = {
        'schema_version': 'syniscopy-e07-training-conditions-v1',
        'condition_set_name': CONDITION_SET_NAME,
        'condition_count': len(CONDITIONS),
        'families': dict(families),
        'conditions': [
            {
                'condition_id': str(c.get('condition_id')),
                'family': str(c.get('approved_family')),
                'name': str(c.get('name')),
                'z_nm': c.get('z_nm'),
                'background_counts': c.get('background_counts'),
                'image_size_px': c.get('image_size_px'),
                'pixel_size_nm': c.get('pixel_size_nm'),
            }
            for c in CONDITIONS
        ],
    }
    (OUTPUT_DIR / 'E07_training_condition_manifest.json').write_text(json.dumps(condition_manifest, indent=2, allow_nan=False), encoding='utf-8')
    print('starting E07 dataset generation...')
    dataset_dir = generate_dataset(
        num_videos=len(CONDITIONS),
        preset_name='default',
        instrument_preset=None,
        base_output_dir=str(OUTPUT_DIR),
        random_seed=RANDOM_SEED,
        resume_existing=RESUME_EXISTING,
        reset_existing=RESET_EXISTING,
        append_on_config_change=False,
        video_param_builder=build_params,
        param_builder_name='E07_caustic_training_conditions',
        verbose=True,
    )
    dataset_dir = Path(dataset_dir)
    dataset_manifest_path = dataset_dir / 'dataset_manifest.json'
    if not dataset_manifest_path.exists():
        raise FileNotFoundError(f'E07 generation finished without dataset_manifest.json at {dataset_manifest_path}')
    print('dataset ready:', dataset_dir)
    print('dataset manifest:', dataset_manifest_path)
else:
    print('generation skipped on local runtime; run this notebook on Colab, or set ALLOW_LOCAL_GENERATION=True deliberately')


In [ ]:
# E07-owned SAM2 VOS derived cache
# E07 owns both the raw Syniscopy dataset and the derived SAM2 VOS cache used by E08.
# The cache is native-size by default; SAM2 training resizes to 1024 during sampling.
if not (IN_COLAB or ALLOW_LOCAL_GENERATION):
    print('E07 SAM2 VOS conversion skipped on local runtime; run this notebook on Colab, or set ALLOW_LOCAL_GENERATION=True deliberately.')
else:
    RAW_DATASET_ROOT = OUTPUT_DIR
    SAM2_DATASET_ROOT = OUTPUT_DIR.parent / 'sam2_vos'
    SUPERVISION_TARGET = 'mask_supported'
    RESET_SAM2_DATASET = bool(RESET_EXISTING)
    SAM2_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    print('E07 raw dataset:', RAW_DATASET_ROOT)
    print('E07 SAM2 VOS cache:', SAM2_DATASET_ROOT)
    # CELL 2: Dataset helpers for Syniscopy supervision masks -> SAM2

    import os
    import cv2
    from pathlib import Path
    import numpy as np
    import json
    from collections import defaultdict

    from dataset_schema import annotation_paths_for_frame, validate_supervision_target

    SUPERVISION_TARGET = validate_supervision_target(SUPERVISION_TARGET)
    print('SAM2 supervision target:', SUPERVISION_TARGET)

    def load_dataset_manifest(dataset_root):
        manifest_path = os.path.join(dataset_root, 'dataset_manifest.json')
        if not os.path.exists(manifest_path):
            raise FileNotFoundError(f'dataset_manifest.json not found at: {manifest_path}')
        with open(manifest_path, 'r') as f:
            manifest = json.load(f)
        if 'videos' not in manifest or not isinstance(manifest['videos'], list):
            raise ValueError("dataset_manifest.json must contain a 'videos' list")
        return manifest

    def load_video_metadata(dataset_root, video_id):
        meta_path = os.path.join(dataset_root, 'metadata', f'{video_id}.json')
        if not os.path.exists(meta_path):
            raise FileNotFoundError(f'Metadata JSON not found for video_id={video_id} at: {meta_path}')
        with open(meta_path, 'r') as f:
            meta = json.load(f)
        if 'num_frames' not in meta:
            raise ValueError(f"Metadata for {video_id} missing 'num_frames' field")
        if 'particles' not in meta or not isinstance(meta['particles'], list):
            raise ValueError(f"Metadata for {video_id} must contain a 'particles' list")
        return meta

    def resolve_frame_sequence_dir(dataset_root, vid_entry):
        video_id = vid_entry.get('video_id')
        frame_rel = vid_entry.get('training_frames_dir') or vid_entry.get('frame_sequence_dir')
        if video_id is None or frame_rel is None:
            raise ValueError(f'Malformed manifest entry missing video_id/frame_sequence_dir: {vid_entry}')
        frame_dir = os.path.join(dataset_root, frame_rel)
        if not os.path.isdir(frame_dir):
            raise FileNotFoundError(f'Lossless frame sequence missing for video_id={video_id}: {frame_dir}')
        return frame_dir

    def load_frame_sequence(frame_dir):
        paths = sorted(Path(frame_dir).glob('*.png'))
        if not paths:
            raise RuntimeError(f'Frame sequence contains zero PNG frames: {frame_dir}')
        frames = []
        for path in paths:
            frame_bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if frame_bgr is None:
                raise RuntimeError(f'Failed to read frame sequence image: {path}')
            if frame_bgr.ndim == 2:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2RGB)
            elif frame_bgr.ndim == 3 and frame_bgr.shape[2] == 3:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            else:
                raise RuntimeError(f'Unsupported frame shape in {path}: {frame_bgr.shape}')
            frames.append(frame_rgb)
        return frames

    def build_video_frame_index(dataset_root, allowed_video_ids=None):
        dataset_root = os.path.abspath(dataset_root)
        manifest = load_dataset_manifest(dataset_root)
        all_entries = []
        videos_info = manifest['videos']
        if allowed_video_ids is not None:
            allowed_video_ids = {str(v) for v in allowed_video_ids}
            videos_info = [v for v in videos_info if str(v.get('video_id')) in allowed_video_ids]
        print(f'Found {len(videos_info)} selected videos in dataset_manifest.json')
        total_target_masks = 0
        missing_target_masks = 0

        for vid_entry in videos_info:
            video_id = vid_entry.get('video_id')
            mask_root_rel = vid_entry.get('mask_root_dir')
            if video_id is None or mask_root_rel is None:
                print(f'Skipping malformed video entry in manifest: {vid_entry}')
                continue

            frame_dir = resolve_frame_sequence_dir(dataset_root, vid_entry)
            mask_root = os.path.join(dataset_root, mask_root_rel)
            schema_path = os.path.join(mask_root, 'annotation_schema.json')
            if not os.path.exists(schema_path):
                raise FileNotFoundError(f'Annotation schema missing for video_id={video_id}: {schema_path}')

            meta = load_video_metadata(dataset_root, video_id)
            num_frames_meta = int(meta['num_frames'])
            particles = meta['particles']
            print(f"\nProcessing video_id={video_id}")
            print(f'  Frame sequence: {frame_dir}')
            print(f'  Mask root: {mask_root}')
            print(f'  Declared num_frames: {num_frames_meta}')
            print(f'  Number of particles: {len(particles)}')

            frames = load_frame_sequence(frame_dir)
            num_frames_video = len(frames)
            if num_frames_video != num_frames_meta:
                print(f'  [Warning] Frame sequence {video_id}: found {num_frames_video} frames, metadata says {num_frames_meta}. Using sequence count.')

            for f_idx in range(num_frames_video):
                frame_img = frames[f_idx]
                target_mask_paths = []
                ignore_mask_paths = []
                loss_weight_paths = []
                for pinfo in particles:
                    if 'particle_index' not in pinfo:
                        continue
                    p_idx = int(pinfo['particle_index'])
                    paths = annotation_paths_for_frame(
                        video_id=video_id,
                        mask_root=mask_root,
                        particle_index=p_idx,
                        frame_index=f_idx,
                        target=SUPERVISION_TARGET,
                    )
                    if paths.target_mask is not None:
                        target_mask_paths.append(str(paths.target_mask))
                        total_target_masks += 1
                    else:
                        missing_target_masks += 1
                    if paths.ignore_mask is not None:
                        ignore_mask_paths.append(str(paths.ignore_mask))
                    if paths.loss_weight is not None:
                        loss_weight_paths.append(str(paths.loss_weight))
                all_entries.append({
                    'video_id': video_id,
                    'frame_index': f_idx,
                    'image': frame_img,
                    'mask_paths': target_mask_paths,
                    'ignore_mask_paths': ignore_mask_paths,
                    'loss_weight_paths': loss_weight_paths,
                })
            print(f'  Collected {num_frames_video} frame entries for video_id={video_id}')

        print(f"\nTotal frame entries across all videos: {len(all_entries)}")
        print(f'Target mask paths found: {total_target_masks}; missing target paths: {missing_target_masks}')
        if all_entries and total_target_masks == 0:
            raise RuntimeError(f'No {SUPERVISION_TARGET} target masks were found. Check mask_output_directory and annotation schema.')
        return all_entries

    def group_entries_by_video(all_entries):
        videos_by_id = defaultdict(list)
        for e in all_entries:
            videos_by_id[e['video_id']].append(e)
        return {vid: sorted(entries, key=lambda x: x['frame_index']) for vid, entries in videos_by_id.items()}

    def _read_binary_union(paths, shape_hw):
        h, w = shape_hw
        union = np.zeros((h, w), dtype=np.uint8)
        for path in paths or []:
            if not os.path.exists(path):
                continue
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            union = np.maximum(union, (m > 0).astype(np.uint8))
        return union

    def _read_loss_weight_union(paths, shape_hw):
        h, w = shape_hw
        if not paths:
            return np.ones((h, w), dtype=np.float32)
        weight = np.zeros((h, w), dtype=np.float32)
        found = False
        for path in paths:
            if not os.path.exists(path):
                continue
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if m is None:
                continue
            found = True
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            weight = np.maximum(weight, m.astype(np.float32) / 255.0)
        return weight if found else np.ones((h, w), dtype=np.float32)

    def build_union_maps_for_entry(entry):
        img = entry['image']
        h, w = img.shape[:2]
        target_mask = _read_binary_union(entry.get('mask_paths', []), (h, w))
        ignore_mask = _read_binary_union(entry.get('ignore_mask_paths', []), (h, w))
        loss_weight = _read_loss_weight_union(entry.get('loss_weight_paths', []), (h, w))
        loss_weight[ignore_mask > 0] = 0.0
        has_fg = np.count_nonzero(target_mask) > 0
        return target_mask, ignore_mask, loss_weight, has_fg

    def build_union_mask_for_entry(entry):
        target_mask, _, _, has_fg = build_union_maps_for_entry(entry)
        return target_mask, has_fg

    print('✅ Dataset helper functions loaded')



    # Select videos with enough lossless frames for E07 SAM2 cache

    import os
    import json
    from pathlib import Path

    DATA_ROOT = str(RAW_DATASET_ROOT)
    MIN_FRAMES = int(SAM2_MIN_VALID_FRAMES)
    TRAINING_FILTER_PATH = os.path.join(str(SAM2_DATASET_ROOT), 'sam2_training_filter.json')

    print('=' * 70)
    print(f'  SELECTING VIDEOS WITH AT LEAST {MIN_FRAMES} LOSSLESS FRAMES')
    print('=' * 70)

    manifest_path = os.path.join(DATA_ROOT, 'dataset_manifest.json')
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

    videos_info = manifest['videos']
    print(f"\nFound {len(videos_info)} videos in manifest")

    eligible_videos = []
    rejected_videos = []

    for vid_entry in videos_info:
        video_id = vid_entry.get('video_id')
        frame_rel = vid_entry.get('training_frames_dir') or vid_entry.get('frame_sequence_dir')
        if video_id is None or frame_rel is None:
            rejected_videos.append({'video_id': video_id, 'reason': 'malformed manifest entry: missing frame_sequence_dir'})
            continue

        frame_dir = Path(DATA_ROOT) / frame_rel
        if not frame_dir.is_dir():
            rejected_videos.append({'video_id': video_id, 'reason': f'could not open frame sequence {frame_dir}'})
            print(f'❌ {video_id:20s}: missing frame sequence')
            continue

        frame_count = len(sorted(frame_dir.glob('*.png')))
        if frame_count >= MIN_FRAMES:
            eligible_videos.append({'video_id': video_id, 'frame_count': frame_count})
            print(f'✅ {video_id:20s}: {frame_count:4d} frames')
        else:
            rejected_videos.append({'video_id': video_id, 'frame_count': frame_count, 'reason': f'< {MIN_FRAMES} frames'})
            print(f'❌ {video_id:20s}: {frame_count:4d} frames')

    filter_doc = {
        'schema_version': 'syniscopy-sam2-training-filter-v1',
        'supervision_target': SUPERVISION_TARGET,
        'min_valid_frames': MIN_FRAMES,
        'sam2_train_num_frames': int(SAM2_TRAIN_NUM_FRAMES),
        'sam2_val_fraction': float(SAM2_VAL_FRACTION),
        'sam2_val_random_seed': int(SAM2_VAL_RANDOM_SEED),
        'sam2_loss_semantics_version': SAM2_LOSS_SEMANTICS_VERSION,
        'eligible_videos': eligible_videos,
        'rejected_videos': rejected_videos,
    }
    with open(TRAINING_FILTER_PATH, 'w') as f:
        json.dump(filter_doc, f, indent=2, allow_nan=False)

    print('\n' + '=' * 70)
    print('  VIDEO SELECTION RESULTS')
    print('=' * 70)
    print('  Original videos:', len(videos_info))
    print('  Eligible:', len(eligible_videos))
    print('  Rejected:', len(rejected_videos))
    print('  Filter file:', TRAINING_FILTER_PATH)
    if len(eligible_videos) == 0:
        raise RuntimeError('No videos are eligible for SAM2 training. Increase generated duration/fps or lower SAM2_MIN_VALID_FRAMES.')


    # Build E07 SAM2-format cache from canonical Syniscopy masks

    import shutil
    from tqdm import tqdm
    import json
    import hashlib
    import random
    import glob

    ROOT = str(RAW_DATASET_ROOT)
    OUT_ROOT = str(SAM2_DATASET_ROOT)

    FRAMES_DIR = os.path.join(OUT_ROOT, 'Frames')
    GT_DIR = os.path.join(OUT_ROOT, 'GT')
    IGNORE_DIR = os.path.join(OUT_ROOT, 'Ignore')
    LOSS_WEIGHT_DIR = os.path.join(OUT_ROOT, 'LossWeight')
    LIST_PATH = os.path.join(OUT_ROOT, 'train_list.txt')
    VAL_LIST_PATH = os.path.join(OUT_ROOT, 'val_list.txt')
    CONVERSION_MANIFEST_PATH = os.path.join(OUT_ROOT, 'conversion_manifest.json')
    TRAINING_FILTER_PATH = os.path.join(OUT_ROOT, 'sam2_training_filter.json')

    MIN_FRAMES = int(SAM2_MIN_VALID_FRAMES)

    def _sha256_file(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def _split_train_val_video_ids(video_ids, val_fraction, seed):
        ids = sorted(video_ids)
        if len(ids) < 2:
            raise RuntimeError("Need at least two converted videos for train/validation splitting.")
        rng = random.Random(int(seed))
        rng.shuffle(ids)
        val_count = max(1, int(round(len(ids) * float(val_fraction))))
        val_count = min(val_count, len(ids) - 1)
        val_ids = sorted(ids[:val_count])
        train_ids = sorted(ids[val_count:])
        return train_ids, val_ids


    SAM2_CONVERSION_CODE_VERSION = 'syniscopy-sam2-vos-conversion-v4-fast-cache-native-size'

    def _eligible_video_ids_from_filter():
        if not os.path.exists(TRAINING_FILTER_PATH):
            return None
        with open(TRAINING_FILTER_PATH, 'r') as f:
            filter_doc = json.load(f)
        eligible = {str(v['video_id']) for v in filter_doc.get('eligible_videos', [])}
        return eligible

    def _input_manifest_signature():
        dataset_manifest_path = os.path.join(ROOT, 'dataset_manifest.json')
        manifest = load_dataset_manifest(ROOT)
        eligible_video_ids = _eligible_video_ids_from_filter()
        selected_video_ids = []
        for entry in manifest['videos']:
            video_id = str(entry.get('video_id'))
            if eligible_video_ids is None or video_id in eligible_video_ids:
                selected_video_ids.append(video_id)
        selected_video_ids = sorted(selected_video_ids)
        selected_digest = hashlib.sha256('\n'.join(selected_video_ids).encode('utf-8')).hexdigest()
        return {
            'dataset_manifest_sha256': _sha256_file(dataset_manifest_path),
            'selected_video_count': len(selected_video_ids),
            'selected_video_ids_sha256': selected_digest,
        }

    def _conversion_config():
        return {
            'schema_version': 'syniscopy-sam2-conversion-v3',
            'conversion_code_version': SAM2_CONVERSION_CODE_VERSION,
            'input_manifest_signature': _input_manifest_signature(),
            'supervision_target': SUPERVISION_TARGET,
            'min_valid_frames': MIN_FRAMES,
            'sam2_val_fraction': float(SAM2_VAL_FRACTION),
            'sam2_val_random_seed': int(SAM2_VAL_RANDOM_SEED),
            'sam2_loss_semantics_version': SAM2_LOSS_SEMANTICS_VERSION,
            'resize_long_side_px': None if SAM2_CONVERSION_RESIZE_LONG_SIDE_PX is None else int(SAM2_CONVERSION_RESIZE_LONG_SIDE_PX),
        }

    def _converted_output_integrity_report(existing):
        kept = list(existing.get('videos_kept', []))
        expected_frames = int(existing.get('frames_kept', 0) or 0)
        problems = []
        converted_frames = 0
        for vid in kept:
            frame_names = {Path(p).stem for p in glob.glob(os.path.join(FRAMES_DIR, vid, '*.jpg'))}
            gt_names = {Path(p).stem for p in glob.glob(os.path.join(GT_DIR, vid, '*.png'))}
            ignore_names = {Path(p).stem for p in glob.glob(os.path.join(IGNORE_DIR, vid, '*.png'))}
            loss_weight_names = {Path(p).stem for p in glob.glob(os.path.join(LOSS_WEIGHT_DIR, vid, '*.png'))}
            converted_frames += len(frame_names)
            if not frame_names:
                problems.append({'video_id': vid, 'reason': 'no converted frames'})
                continue
            if not (frame_names == gt_names == ignore_names == loss_weight_names):
                problems.append({
                    'video_id': vid,
                    'reason': 'converted filename mismatch',
                    'frames': len(frame_names),
                    'gt': len(gt_names),
                    'ignore': len(ignore_names),
                    'loss_weight': len(loss_weight_names),
                })
        if expected_frames > 0 and converted_frames != expected_frames:
            problems.append({
                'reason': 'converted frame count mismatch',
                'expected_frames': expected_frames,
                'converted_frames': converted_frames,
            })
        return {
            'ok': not problems and bool(kept) and expected_frames > 0,
            'videos_checked': len(kept),
            'expected_frames': expected_frames,
            'converted_frames': converted_frames,
            'problem_count': len(problems),
            'problems': problems[:10],
        }

    def _conversion_config_equivalent(existing_config, current_config):
        def _normalized(doc):
            doc = dict(doc or {})
            signature = dict(doc.get('input_manifest_signature') or {})
            signature.pop('training_filter_sha256', None)
            signature.pop('eligible_filter_present', None)
            doc['input_manifest_signature'] = signature
            return doc
        return _normalized(existing_config) == _normalized(current_config)

    def _existing_conversion_is_current(config):
        if RESET_SAM2_DATASET:
            return False
        required_paths = [
            CONVERSION_MANIFEST_PATH,
            LIST_PATH,
            VAL_LIST_PATH,
            FRAMES_DIR,
            GT_DIR,
            IGNORE_DIR,
            LOSS_WEIGHT_DIR,
        ]
        if not all(os.path.exists(path) for path in required_paths):
            return False
        try:
            with open(CONVERSION_MANIFEST_PATH, 'r') as f:
                existing = json.load(f)
        except Exception:
            return False
        if not _conversion_config_equivalent(existing.get('conversion_config'), config):
            return False
        with open(LIST_PATH) as f:
            train_video_ids = [line.strip() for line in f if line.strip()]
        with open(VAL_LIST_PATH) as f:
            val_video_ids = [line.strip() for line in f if line.strip()]
        if not train_video_ids or not val_video_ids:
            return False
        integrity = _converted_output_integrity_report(existing)
        if not integrity['ok']:
            print('Cached SAM2 dataset conversion is incomplete or inconsistent:')
            print(json.dumps(integrity, indent=2))
            return False
        globals()['SAM2_CONVERSION_INTEGRITY'] = integrity
        return True

    config = _conversion_config()
    if _existing_conversion_is_current(config):
        print('✅ Current SAM2 dataset already exists. Skipping rebuild.')
        with open(LIST_PATH) as f:
            _train_ids = [line.strip() for line in f if line.strip()]
        with open(VAL_LIST_PATH) as f:
            _val_ids = [line.strip() for line in f if line.strip()]
        print('Using cached data at:', OUT_ROOT)
        print('   Training videos:', len(_train_ids))
        print('   Validation videos:', len(_val_ids))
        if 'SAM2_CONVERSION_INTEGRITY' in globals():
            print('   Converted frames checked:', SAM2_CONVERSION_INTEGRITY.get('converted_frames'))
    else:
        if os.path.exists(OUT_ROOT):
            print('Removing cached SAM2 dataset conversion...')
            shutil.rmtree(OUT_ROOT)

        print('Building SAM2-format dataset with:')
        print('  supervision_target:', SUPERVISION_TARGET)
        print('  min_valid_frames:', MIN_FRAMES)
        print('  validation_fraction:', SAM2_VAL_FRACTION)
        print('  validation_seed:', SAM2_VAL_RANDOM_SEED)
        print('  output:', OUT_ROOT)
        print('  resize_long_side_px:', 'native' if SAM2_CONVERSION_RESIZE_LONG_SIDE_PX is None else SAM2_CONVERSION_RESIZE_LONG_SIDE_PX)

        os.makedirs(FRAMES_DIR, exist_ok=True)
        os.makedirs(GT_DIR, exist_ok=True)
        os.makedirs(IGNORE_DIR, exist_ok=True)
        os.makedirs(LOSS_WEIGHT_DIR, exist_ok=True)

        eligible_video_ids = _eligible_video_ids_from_filter()
        if eligible_video_ids is not None:
            print('Using SAM2 training filter:', TRAINING_FILTER_PATH)

        all_entries = build_video_frame_index(ROOT, allowed_video_ids=eligible_video_ids)

        videos_by_id = group_entries_by_video(all_entries)
        video_ids = sorted(videos_by_id.keys())
        kept_video_ids = []
        total_kept = 0
        total_skipped_empty = 0
        rejected = []

        for vid in tqdm(video_ids, desc='Converting videos'):
            frames = videos_by_id[vid]
            if len(frames) < MIN_FRAMES:
                rejected.append({'video_id': vid, 'reason': f'{len(frames)} decoded frames < {MIN_FRAMES}'})
                continue

            out_img_dir = os.path.join(FRAMES_DIR, vid)
            out_gt_dir = os.path.join(GT_DIR, vid)
            out_ignore_dir = os.path.join(IGNORE_DIR, vid)
            out_loss_weight_dir = os.path.join(LOSS_WEIGHT_DIR, vid)
            os.makedirs(out_img_dir, exist_ok=True)
            os.makedirs(out_gt_dir, exist_ok=True)
            os.makedirs(out_ignore_dir, exist_ok=True)
            os.makedirs(out_loss_weight_dir, exist_ok=True)

            valid_frames = 0
            for entry in frames:
                frame_idx = entry['frame_index']
                img = entry['image']
                union_mask, ignore_mask, loss_weight_map, has_fg = build_union_maps_for_entry(entry)
                if not has_fg or union_mask.max() == 0:
                    total_skipped_empty += 1
                    continue

                if SAM2_CONVERSION_RESIZE_LONG_SIDE_PX is None:
                    img_resized = img
                    mask_resized = union_mask
                    ignore_resized = ignore_mask
                    loss_weight_resized = loss_weight_map
                else:
                    h, w = img.shape[:2]
                    r = float(SAM2_CONVERSION_RESIZE_LONG_SIDE_PX) / max(h, w)
                    new_w = max(1, int(round(w * r)))
                    new_h = max(1, int(round(h * r)))
                    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
                    mask_resized = cv2.resize(union_mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
                    ignore_resized = cv2.resize(ignore_mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
                    loss_weight_resized = cv2.resize(loss_weight_map, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

                cv2.imwrite(os.path.join(out_img_dir, f'{frame_idx:05d}.jpg'), cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR))
                cv2.imwrite(os.path.join(out_gt_dir, f'{frame_idx:05d}.png'), (mask_resized > 0).astype(np.uint8) * 255)
                cv2.imwrite(os.path.join(out_ignore_dir, f'{frame_idx:05d}.png'), (ignore_resized > 0).astype(np.uint8) * 255)
                cv2.imwrite(os.path.join(out_loss_weight_dir, f'{frame_idx:05d}.png'), np.clip(loss_weight_resized * 255.0, 0, 255).astype(np.uint8))

                valid_frames += 1
                total_kept += 1

            if valid_frames >= MIN_FRAMES:
                kept_video_ids.append(vid)
            else:
                shutil.rmtree(out_img_dir, ignore_errors=True)
                shutil.rmtree(out_gt_dir, ignore_errors=True)
                shutil.rmtree(out_ignore_dir, ignore_errors=True)
                shutil.rmtree(out_loss_weight_dir, ignore_errors=True)
                rejected.append({'video_id': vid, 'reason': f'{valid_frames} supervised frames < {MIN_FRAMES}'})

        train_video_ids, val_video_ids = _split_train_val_video_ids(
            kept_video_ids, SAM2_VAL_FRACTION, SAM2_VAL_RANDOM_SEED
        )

        with open(LIST_PATH, 'w') as f:
            for vid in train_video_ids:
                f.write(vid + '\n')
        with open(VAL_LIST_PATH, 'w') as f:
            for vid in val_video_ids:
                f.write(vid + '\n')

        conversion_manifest = {
            'conversion_config': config,
            'videos_kept': kept_video_ids,
            'train_videos': train_video_ids,
            'val_videos': val_video_ids,
            'frames_kept': total_kept,
            'frames_skipped_empty': total_skipped_empty,
            'rejected': rejected,
            'frames_dir': FRAMES_DIR,
            'gt_dir': GT_DIR,
            'ignore_dir': IGNORE_DIR,
            'loss_weight_dir': LOSS_WEIGHT_DIR,
            'train_list_txt': LIST_PATH,
            'val_list_txt': VAL_LIST_PATH,
        }
        with open(CONVERSION_MANIFEST_PATH, 'w') as f:
            json.dump(conversion_manifest, f, indent=2, allow_nan=False)

        print('\n✅ Conversion complete.')
        print('   Videos kept:', len(kept_video_ids))
        print('   Training videos:', len(train_video_ids))
        print('   Validation videos:', len(val_video_ids))
        print('   Frames kept:', total_kept)
        print('   Frames skipped because target mask was empty:', total_skipped_empty)
        print('   Conversion manifest:', CONVERSION_MANIFEST_PATH)

        if len(kept_video_ids) == 0:
            raise RuntimeError('SAM2 conversion produced zero training videos. Check supervision target, masks, and generation duration.')

        audit_path = os.path.join(ROOT, 'supervision_audit.json')
        if os.path.exists(audit_path):
            with open(audit_path, 'r') as f:
                audit = json.load(f)
            print('\nSupervision audit summary:')
            for key in ['ignored_particle_fraction', 'ignored_pixel_fraction', 'positive_supervision_pixel_fraction', 'low_signal_particle_fraction', 'high_crlb_particle_fraction']:
                if key in audit:
                    print(f'   {key}: {audit[key]}')
